In [ ]:
# ====================================================================================================
# IMPUTAÇÃO MICE COM MICEFOREST (RANDOM FOREST) - DADOS MISTOS (CONTÍNUAS + ORDINAIS)
# DATASET: TIPS (Restaurant Tips) - 2 contínuas + 5 ordinais
# ====================================================================================================
# 1. Variáveis contínuas: total_bill, tip
# 2. Variáveis ordinais: size, day, time, smoker, sex
# 3. Cálculo de SMAE específico para contínuas (total_bill, tip) e para ordinais (5 vars)
# 4. CORREÇÃO DO SMAE: calcula-se a razão (erro_imp / erro_referência) para cada coluna,
#    depois a média dessas razões entre as colunas (dentro de cada imputação).
#    As médias entre colunas de cada imputação são então combinadas via regras de Rubin.
# 5. Missing MCAR estratificado por coluna (mesma proporção em cada variável).
# ====================================================================================================

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import random
import sys
import time
import os
from IPython.display import display, HTML
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                           cohen_kappa_score, matthews_corrcoef, balanced_accuracy_score,
                           mean_squared_error, mean_absolute_error, r2_score)
from scipy.stats import t, pearsonr

# Importa o miceforest (biblioteca moderna para MICE)
try:
    import miceforest as mf
    print("✅ miceforest carregado com sucesso.")
except ImportError:
    display(HTML('<div class="warning-box">📦 Instalando miceforest...</div>'))
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "miceforest", "--quiet"])
    import miceforest as mf
    print("✅ miceforest instalado e carregado.")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

TEMPO_GLOBAL_INICIO = time.perf_counter()

# ====================================================================================================
# ESTILOS CSS (mantidos do original)
# ====================================================================================================
CSS_STYLES = """
<style>
    .header {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        padding: 15px;
        border-radius: 10px;
        margin: 10px 0;
        text-align: center;
        font-weight: bold;
        font-size: 18px;
    }
    .success-box { background-color: #d4edda; color: #155724; padding: 12px; border-radius: 5px; margin: 10px 0; font-weight: bold; }
    .warning-box { background-color: #fff3cd; color: #856404; padding: 12px; border-radius: 5px; margin: 10px 0; font-weight: bold; }
    .info-box { background-color: #d1ecf1; color: #0c5460; padding: 12px; border-radius: 5px; margin: 10px 0; }
    .iteration-box { background-color: #e8f4fd; border-left: 5px solid #3498db; padding: 10px; margin: 10px 0; border-radius: 5px; font-family: monospace; }
    .metrics-highlight { background: linear-gradient(135deg, #ffd89b 0%, #19547b 100%); color: white; padding: 15px; border-radius: 10px; margin: 10px 0; text-align: center; font-weight: bold; }
    .dataframe-table { font-family: Arial; border-collapse: collapse; width: 100%; box-shadow: 0 4px 8px rgba(0,0,0,0.1); border-radius: 10px; overflow: hidden; margin: 20px 0; }
    .dataframe-table th { background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%); color: white; padding: 12px; text-align: left; }
    .dataframe-table td { padding: 10px; border-bottom: 1px solid #ddd; }
    .dataframe-table tr:nth-child(even) { background-color: #f8f9fa; }
    .dataframe-table tr:hover { background-color: #e9ecef; }
    .miceforest-badge { background: linear-gradient(135deg, #FF6B6B 0%, #4ECDC4 100%); color: white; padding: 5px 10px; border-radius: 15px; font-size: 12px; display: inline-block; margin: 2px; }
</style>
"""
display(HTML(CSS_STYLES))

# ====================================================================================================
# CARREGAMENTO DO DATASET TIPS
# ====================================================================================================
display(HTML('<div class="header">🍽️ CARREGANDO DATASET TIPS (2 contínuas + 5 ordinais)</div>'))

# Carrega o dataset tips do seaborn
import seaborn as sns
tips = sns.load_dataset("tips")
df_raw = tips.copy()

# Definição das variáveis
variaveis_continuas = ['total_bill', 'tip']
variaveis_ordinais = ['size', 'day', 'time', 'smoker', 'sex']  # todas tratadas como ordinais
colunas_categoricas = variaveis_ordinais
all_vars = variaveis_continuas + colunas_categoricas

df_original = df_raw[all_vars].copy().dropna().reset_index(drop=True)

display(HTML(f'<div class="success-box">✅ Dados carregados: {df_original.shape[0]} linhas, {df_original.shape[1]} colunas.</div>'))

display(HTML(f'''
<div class="info-box">
    <b>Configuração dos tipos:</b><br>
    • Contínuas: {', '.join(variaveis_continuas)}<br>
    • Ordinais: {', '.join(variaveis_ordinais)}<br>
    • Amostras após limpeza: {df_original.shape[0]}
</div>
'''))

# Converte tipos: categóricas para string (para manter os labels), contínuas para numérico
df_work = df_original.copy()
for col in colunas_categoricas:
    df_work[col] = df_work[col].astype(str)
for col in variaveis_continuas:
    df_work[col] = pd.to_numeric(df_work[col], errors='coerce')
df_work = df_work.dropna().reset_index(drop=True)

# ====================================================================================================
# FUNÇÕES AUXILIARES (CRIAÇÃO DE MISSING ESTRATIFICADO, ENCODERS SIMPLES, MÉTRICAS)
# ====================================================================================================
def criar_missing_mcar_estratificado(df, percentual=0.15, random_seed=42):
    """
    Insere missing MCAR de forma ESTRATIFICADA por coluna.
    Garante que cada coluna tenha exatamente a mesma proporção de valores ausentes.
    """
    np.random.seed(random_seed)
    random.seed(random_seed)
    df_com_na = df.copy()
    posicoes_originais = {}
    for col in df.columns:
        n_missing = int(len(df) * percentual)
        indices = np.random.choice(df.index, size=n_missing, replace=False)
        for i in indices:
            posicoes_originais[(i, col)] = df.loc[i, col]
            df_com_na.loc[i, col] = np.nan
    return df_com_na, posicoes_originais

def preparar_encoders_e_dados_simples(df_missing, colunas_categoricas, colunas_continuas):
    """
    Prepara LabelEncoders para todas as categóricas.
    Retorna df_label (numérico), label_encoders, e objetos vazios para compatibilidade.
    """
    label_encoders = {}
    df_label = pd.DataFrame(index=df_missing.index)

    for col in colunas_categoricas:
        le = LabelEncoder()
        valores_obs = df_missing[col].dropna()
        if len(valores_obs) > 0:
            le.fit(valores_obs.astype(str))
        else:
            le.fit(['desconhecido'])
        label_encoders[col] = le

        col_cod = np.full(len(df_missing), np.nan)
        mask_obs = df_missing[col].notna()
        if mask_obs.any():
            col_cod[mask_obs] = le.transform(df_missing.loc[mask_obs, col].astype(str))
        df_label[col] = col_cod

    for col in colunas_continuas:
        df_label[col] = df_missing[col].copy()

    # Objetos vazios para manter compatibilidade (não usados)
    df_onehot = pd.DataFrame(index=df_missing.index)
    onehot_encoders = {}
    normalizadores = {}

    return df_label, df_onehot, label_encoders, onehot_encoders, normalizadores

# ====================================================================================================
# IMPUTAÇÃO COM MICEFOREST (CORRIGIDA PARA VERSÃO 6.x)
# ====================================================================================================
def imputar_com_miceforest(df_label, random_seed=42, iterations=5):
    """
    Usa miceforest para imputar valores faltantes (df_label já é numérico).
    Compatível com miceforest >= 6.0.
    """
    kernel = mf.ImputationKernel(
        df_label,
        num_datasets=1,          # argumento correto (substitui 'datasets')
        random_state=random_seed
    )
    kernel.mice(iterations=iterations, verbose=False)
    df_imputado = kernel.complete_data(dataset=0)
    return df_imputado

# ====================================================================================================
# MÉTRICAS (com SMAE corrigido)
# ====================================================================================================
def smae_continuous(y_true, y_pred):
    """
    SMAE para variável contínua (baseado em Zhao & Udell 2020).
    Soma(|imputado - real|) / Soma(|mediana(real) - real|)
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mediana = np.median(y_true)
    num = np.sum(np.abs(y_true - y_pred))
    den = np.sum(np.abs(y_true - mediana))
    if den == 0:
        return 0.0 if num == 0 else np.inf
    return num / den

def smae_categorical(y_true, y_pred):
    """
    SMAE para variável ordinal/categórica (baseado em Zhao & Udell 2020).
    Em vez de mediana, usa moda (categoria mais frequente).
    Soma(indicador(imputado != real)) / Soma(indicador(moda != real))
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    # Encontra a moda (categoria mais frequente)
    vals, counts = np.unique(y_true, return_counts=True)
    moda = vals[np.argmax(counts)]
    num = np.sum(y_true != y_pred)
    den = np.sum(y_true != moda)
    if den == 0:
        return 0.0 if num == 0 else np.inf
    return num / den

def metricas_continuas_avancadas(reais, imputados):
    reais = np.array(reais); imputados = np.array(imputados)
    mae = mean_absolute_error(reais, imputados)
    rng = reais.max() - reais.min()
    nmae = mae / rng if rng > 0 else np.nan
    mse = mean_squared_error(reais, imputados)
    mape = np.mean(np.abs((reais - imputados) / (reais + 1e-8))) * 100
    smape = 2.0 * np.mean(np.abs(reais - imputados) / (np.abs(reais) + np.abs(imputados) + 1e-8)) * 100
    if len(reais) > 1 and np.std(reais) > 0 and np.std(imputados) > 0:
        corr, _ = pearsonr(reais, imputados)
    else:
        corr = np.nan
    rmse = np.sqrt(mse); nrmse = rmse / rng if rng > 0 else np.nan
    r2 = r2_score(reais, imputados)
    smae = smae_continuous(reais, imputados)
    return {'MAE':mae,'NMAE':nmae,'MSE':mse,'MAPE':mape,'sMAPE':smape,'Correlacao':corr,'RMSE':rmse,'NRMSE':nrmse,'R2':r2,'SMAE':smae}

def metricas_categoricas_avancadas(reais, imputados):
    if len(reais)==0: return {k:np.nan for k in ['Acurácia','Precisão','Recall','F1','Kappa','MCC','SMAE']}
    reais = np.array(reais); imputados = np.array(imputados)
    acc = accuracy_score(reais, imputados)
    prec = precision_score(reais, imputados, average='macro', zero_division=0)
    rec = recall_score(reais, imputados, average='macro', zero_division=0)
    f1 = f1_score(reais, imputados, average='macro', zero_division=0)
    kappa = cohen_kappa_score(reais, imputados)
    mcc = matthews_corrcoef(reais, imputados)
    smae = smae_categorical(reais, imputados)
    return {'Acurácia':acc,'Precisão':prec,'Recall':rec,'F1':f1,'Kappa':kappa,'MCC':mcc,'SMAE':smae}

def gower_distance_mixed(df_real, df_imp, col_cont, col_cat, cat_encoders=None):
    n = len(df_real)
    if n == 0: return np.nan
    ranges = {}
    for col in col_cont:
        min_val = df_real[col].min(); max_val = df_real[col].max()
        ranges[col] = (max_val - min_val) if (max_val - min_val) > 0 else 1.0
    dist_total = 0.0
    for i in range(n):
        row_dist = 0.0
        for col in col_cont:
            d = np.abs(df_real.iloc[i][col] - df_imp.iloc[i][col]) / ranges[col]
            row_dist += d
        for col in col_cat:
            if cat_encoders and col in cat_encoders:
                le = cat_encoders[col]
                true_val = df_real.iloc[i][col]
                imp_val = df_imp.iloc[i][col]
                if isinstance(true_val, str):
                    true_code = le.transform([true_val])[0] if true_val in le.classes_ else -1
                else:
                    true_code = true_val if true_val in le.classes_ else -1
                if isinstance(imp_val, str):
                    imp_code = le.transform([imp_val])[0] if imp_val in le.classes_ else -1
                else:
                    imp_code = imp_val if imp_val in le.classes_ else -1
                d = 0 if true_code == imp_code else 1
            else:
                d = 0 if df_real.iloc[i][col] == df_imp.iloc[i][col] else 1
            row_dist += d
        dist_total += row_dist / (len(col_cont) + len(col_cat))
    return dist_total / n

def correlation_matrix_error(df_real, df_imp, col_cont):
    if len(col_cont) < 2: return np.nan
    real_corr = df_real[col_cont].corr().values
    imp_corr = df_imp[col_cont].corr().values
    frob_diff = np.linalg.norm(real_corr - imp_corr, 'fro')
    frob_real = np.linalg.norm(real_corr, 'fro')
    if frob_real == 0: return 0.0 if frob_diff == 0 else np.inf
    return frob_diff / frob_real

# ====================================================================================================
# MÚLTIPLAS IMPUTAÇÕES + REGRAS DE RUBIN (com SMAE específico corrigido e missing estratificado)
# ====================================================================================================
def multiple_imputation_rubin_miceforest(df_clean_features, proporcao_missing,
                                         col_cont_todas,   # todas as contínuas (total_bill, tip)
                                         col_cont_smae,    # contínuas para SMAE (mesmo conjunto)
                                         col_cat_todas,    # todas as ordinais (size, day, time, smoker, sex)
                                         m=3, random_seed=RANDOM_SEED, verbose=False):
    """
    Realiza múltiplas imputações com miceforest e combina via regras de Rubin.
    Utiliza missing estratificado por coluna.
    Retorna dicionário com estimativas combinadas:
        - Métricas para todas as contínuas (MAE, RMSE, etc.)
        - SMAE específico para contínuas (col_cont_smae)
        - Métricas para todas as categóricas (Acurácia, etc.)
        - SMAE específico para ordinais (col_cat_todas)
        - Gower, erro matriz correlação
    """
    cont_estimativas = {met: [] for met in ['MAE','NMAE','MSE','MAPE','sMAPE','Correlacao','RMSE','NRMSE','R2']}  # sem SMAE geral
    cat_estimativas = {met: [] for met in ['Acurácia','Precisão','Recall','F1','Kappa','MCC']}  # sem SMAE geral
    gower_estimativas = []
    corr_err_estimativas = []
    smae_cont_estimativas = []   # SMAE específico para contínuas (lista)
    smae_ord_estimativas = []    # SMAE específico para ordinais

    for i in range(m):
        if verbose:
            display(HTML(f'<div class="iteration-box">🔹 Imputação {i+1}/{m} com miceforest</div>'))
        seed = random_seed + i*1000
        # Usa a nova função estratificada
        df_missing, pos_orig = criar_missing_mcar_estratificado(df_clean_features, percentual=proporcao_missing, random_seed=seed)

        df_label, _, label_encoders, _, _ = preparar_encoders_e_dados_simples(
            df_missing, col_cat_todas, col_cont_todas
        )

        df_imputado = imputar_com_miceforest(df_label, random_seed=seed, iterations=5)

        # Coleta dos valores reais e imputados apenas nas posições com missing
        real_cont = {col: [] for col in col_cont_todas}
        imp_cont = {col: [] for col in col_cont_todas}
        real_cat = {col: [] for col in col_cat_todas}
        imp_cat = {col: [] for col in col_cat_todas}

        for (linha, col), real in pos_orig.items():
            imp = df_imputado.loc[linha, col]
            if col in col_cont_todas:
                real_cont[col].append(real)
                imp_cont[col].append(imp)
            else:
                if col in label_encoders:
                    try:
                        imp_str = label_encoders[col].inverse_transform([int(round(imp))])[0]
                    except:
                        imp_str = str(imp)
                else:
                    imp_str = str(imp)
                real_cat[col].append(str(real))
                imp_cat[col].append(imp_str)

        # Métricas para todas as contínuas (MAE, RMSE, etc.)
        cont_metrics_list = []
        for col in col_cont_todas:
            if real_cont[col]:
                cont_metrics_list.append(metricas_continuas_avancadas(real_cont[col], imp_cont[col]))
        if cont_metrics_list:
            for met in cont_estimativas.keys():
                vals = [d[met] for d in cont_metrics_list if not np.isnan(d[met])]
                media = np.mean(vals) if vals else np.nan
                cont_estimativas[met].append(media)
        else:
            for met in cont_estimativas.keys():
                cont_estimativas[met].append(np.nan)

        # SMAE específico para contínuas (apenas col_cont_smae) - CORREÇÃO APLICADA
        # Calcula a razão por coluna e depois média entre colunas
        smae_cont_values = []
        for col in col_cont_smae:
            if real_cont[col]:
                smae_cont_values.append(smae_continuous(real_cont[col], imp_cont[col]))
        smae_cont_medio = np.nanmean(smae_cont_values) if smae_cont_values else np.nan
        smae_cont_estimativas.append(smae_cont_medio)

        # Métricas para todas as categóricas (Acurácia, etc.)
        cat_metrics_list = []
        for col in col_cat_todas:
            if real_cat[col]:
                cat_metrics_list.append(metricas_categoricas_avancadas(real_cat[col], imp_cat[col]))
        if cat_metrics_list:
            for met in cat_estimativas.keys():
                vals = [d[met] for d in cat_metrics_list if not np.isnan(d[met])]
                media = np.mean(vals) if vals else np.nan
                cat_estimativas[met].append(media)
        else:
            for met in cat_estimativas.keys():
                cat_estimativas[met].append(np.nan)

        # SMAE específico para ordinais (todas as col_cat_todas) - CORREÇÃO APLICADA
        smae_ord_values = []
        for col in col_cat_todas:
            if real_cat[col]:
                smae_ord_values.append(smae_categorical(real_cat[col], imp_cat[col]))
        smae_ord_medio = np.nanmean(smae_ord_values) if smae_ord_values else np.nan
        smae_ord_estimativas.append(smae_ord_medio)

        # Gower distance
        linhas_com_missing = sorted(set([l for (l, _) in pos_orig.keys()]))
        if linhas_com_missing:
            df_real_gower = df_clean_features.loc[linhas_com_missing].copy()
            df_imp_gower = pd.DataFrame(index=linhas_com_missing)
            for col in col_cont_todas:
                df_imp_gower[col] = df_imputado.loc[linhas_com_missing, col].values
            for col in col_cat_todas:
                le = label_encoders[col]
                codes = df_imputado.loc[linhas_com_missing, col].astype(int).values
                df_imp_gower[col] = le.inverse_transform(codes)
            gower = gower_distance_mixed(df_real_gower, df_imp_gower, col_cont_todas, col_cat_todas, cat_encoders=label_encoders)
            gower_estimativas.append(gower)
        else:
            gower_estimativas.append(np.nan)

        # Erro matriz correlação
        if len(col_cont_todas) >= 2:
            df_real_full = df_clean_features[col_cont_todas]
            df_imp_full = pd.DataFrame(index=df_imputado.index)
            for col in col_cont_todas:
                df_imp_full[col] = df_imputado[col].values
            corr_err = correlation_matrix_error(df_real_full, df_imp_full, col_cont_todas)
            corr_err_estimativas.append(corr_err)
        else:
            corr_err_estimativas.append(np.nan)

    # Função de combinação de Rubin
    def rubin_combine(vals):
        vals = [v for v in vals if not np.isnan(v)]
        if len(vals) == 0:
            return {'Q_bar': np.nan, 'se': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan, 'std_between': np.nan}
        Q_bar = np.mean(vals)
        m_eff = len(vals)
        B = np.var(vals, ddof=1) if m_eff > 1 else 0
        T = (1 + 1/m) * B
        se = np.sqrt(T)
        if B > 0:
            df = (m_eff - 1) * (1 + (0 / ((1 + 1/m) * B)))**2
        else:
            df = 1000
        tval = t.ppf(0.975, df)
        ci_low = Q_bar - tval * se
        ci_high = Q_bar + tval * se
        return {'Q_bar': Q_bar, 'se': se, 'ci_lower': ci_low, 'ci_upper': ci_high, 'std_between': np.sqrt(B)}

    resultados = {}
    for met in cont_estimativas:
        resultados[f'cont_{met}'] = rubin_combine(cont_estimativas[met])
    for met in cat_estimativas:
        resultados[f'cat_{met}'] = rubin_combine(cat_estimativas[met])
    resultados['smae_cont'] = rubin_combine(smae_cont_estimativas)
    resultados['smae_ord'] = rubin_combine(smae_ord_estimativas)
    resultados['gower'] = rubin_combine(gower_estimativas)
    resultados['corr_matrix_error'] = rubin_combine(corr_err_estimativas)
    return resultados

# ====================================================================================================
# BOOTSTRAP EXTERNO
# ====================================================================================================
def bootstrap_misto(df_clean_features, proporcao_missing,
                    col_cont_todas, col_cont_smae, col_cat_todas,
                    B=10, m=3, random_seed=RANDOM_SEED, verbose_first=True):
    boot_cont = {met: [] for met in ['MAE','NMAE','MSE','MAPE','sMAPE','Correlacao','RMSE','NRMSE','R2']}
    boot_cat = {met: [] for met in ['Acurácia','Precisão','Recall','F1','Kappa','MCC']}
    boot_gower = []
    boot_corr_err = []
    boot_smae_cont = []
    boot_smae_ord = []
    tempos = []

    for b in range(B):
        start_b = time.perf_counter()
        print(f"\n--- Bootstrap {b+1}/{B} (NA={proporcao_missing*100:.0f}%) ---")
        n = len(df_clean_features)
        idx = np.random.choice(n, size=n, replace=True)
        df_boot = df_clean_features.iloc[idx].reset_index(drop=True)
        verbose_this = (verbose_first and b == 0)
        rubin_results = multiple_imputation_rubin_miceforest(
            df_boot, proporcao_missing,
            col_cont_todas, col_cont_smae, col_cat_todas,
            m=m, random_seed=random_seed+b*100, verbose=verbose_this
        )
        for met in boot_cont:
            boot_cont[met].append(rubin_results[f'cont_{met}']['Q_bar'])
        for met in boot_cat:
            boot_cat[met].append(rubin_results[f'cat_{met}']['Q_bar'])
        boot_gower.append(rubin_results['gower']['Q_bar'])
        boot_corr_err.append(rubin_results['corr_matrix_error']['Q_bar'])
        boot_smae_cont.append(rubin_results['smae_cont']['Q_bar'])
        boot_smae_ord.append(rubin_results['smae_ord']['Q_bar'])
        elapsed = time.perf_counter() - start_b
        tempos.append(elapsed)
        print(f"   Tempo: {elapsed:.2f}s | Acurácia: {boot_cat['Acurácia'][-1]:.3f} | RMSE: {boot_cont['RMSE'][-1]:.3f}")

    def bootstrap_stats(vals):
        vals = [v for v in vals if not np.isnan(v)]
        if not vals:
            return {'media': np.nan, 'ic_inf': np.nan, 'ic_sup': np.nan}
        return {'media': np.mean(vals), 'ic_inf': np.percentile(vals, 2.5), 'ic_sup': np.percentile(vals, 97.5)}

    resultados_finais = {}
    for met in boot_cont:
        resultados_finais[f'cont_{met}'] = bootstrap_stats(boot_cont[met])
    for met in boot_cat:
        resultados_finais[f'cat_{met}'] = bootstrap_stats(boot_cat[met])
    resultados_finais['gower'] = bootstrap_stats(boot_gower)
    resultados_finais['corr_matrix_error'] = bootstrap_stats(boot_corr_err)
    resultados_finais['smae_cont'] = bootstrap_stats(boot_smae_cont)
    resultados_finais['smae_ord'] = bootstrap_stats(boot_smae_ord)
    tempo_medio = np.mean(tempos)
    tempo_total = sum(tempos)
    return resultados_finais, tempo_total, tempo_medio

# ====================================================================================================
# EXECUÇÃO PRINCIPAL
# ====================================================================================================
display(HTML('<div class="header">🔬 AVALIAÇÃO COM BOOTSTRAP EXTERNO + RUBIN (MiceForest) - TIPS</div>'))

proporcoes_teste = [0.15, 0.30, 0.40, 0.60, 0.75, 0.90]
B_BOOTSTRAP = 100   # número de repetições bootstrap (ajuste conforme demanda)
M_IMPUTACOES = 3  # número de imputações múltiplas

# Definição das listas de colunas
col_cont_todas = ['total_bill', 'tip']        # todas as contínuas
col_cont_smae = ['total_bill', 'tip']         # contínuas para SMAE (mesmas)
col_cat_todas = ['size', 'day', 'time', 'smoker', 'sex']   # ordinais

tabela_cont = []
tabela_cat = []
tabela_extra = []

for prop in proporcoes_teste:
    pct = prop*100
    display(HTML(f'<div class="metrics-highlight">🎯 PROPORÇÃO: {pct:.0f}%</div>'))
    res, tempo_total, tempo_medio = bootstrap_misto(
        df_work, prop,
        col_cont_todas=col_cont_todas,
        col_cont_smae=col_cont_smae,
        col_cat_todas=col_cat_todas,
        B=B_BOOTSTRAP,
        m=M_IMPUTACOES,
        random_seed=RANDOM_SEED,
        verbose_first=(prop==proporcoes_teste[0])
    )
    # Linha contínuas (métricas tradicionais)
    linha_cont = {'Proporção': f'{pct:.0f}%'}
    for met in ['MAE','NMAE','MSE','MAPE','sMAPE','Correlacao','RMSE','NRMSE','R2']:
        stats = res[f'cont_{met}']
        linha_cont[met] = f"{stats['media']:.4f} [{stats['ic_inf']:.4f}–{stats['ic_sup']:.4f}]" if not np.isnan(stats['media']) else 'NaN'
    # Adiciona SMAE contínuo específico
    smae_c = res['smae_cont']
    linha_cont['SMAE_cont'] = f"{smae_c['media']:.4f} [{smae_c['ic_inf']:.4f}–{smae_c['ic_sup']:.4f}]"
    tabela_cont.append(linha_cont)

    # Linha categóricas (métricas tradicionais)
    linha_cat = {'Proporção': f'{pct:.0f}%'}
    for met in ['Acurácia','Precisão','Recall','F1','Kappa','MCC']:
        stats = res[f'cat_{met}']
        linha_cat[met] = f"{stats['media']:.2%} [{stats['ic_inf']:.2%}–{stats['ic_sup']:.2%}]"
    # Adiciona SMAE ordinal específico
    smae_o = res['smae_ord']
    linha_cat['SMAE_ord'] = f"{smae_o['media']:.4f} [{smae_o['ic_inf']:.4f}–{smae_o['ic_sup']:.4f}]"
    tabela_cat.append(linha_cat)

    # Linha extra (Gower, erro matriz correlação)
    gow = res['gower']
    corr_err = res['corr_matrix_error']
    linha_extra = {
        'Proporção': f'{pct:.0f}%',
        'Gower': f"{gow['media']:.4f} [{gow['ic_inf']:.4f}–{gow['ic_sup']:.4f}]",
        'Erro_mat_corr': f"{corr_err['media']:.4f} [{corr_err['ic_inf']:.4f}–{corr_err['ic_sup']:.4f}]"
    }
    tabela_extra.append(linha_extra)

    display(HTML(f'<div class="info-box">⏱️ Tempo total bootstrap: {tempo_total:.2f}s | Média por boot: {tempo_medio:.2f}s</div>'))

# Exibir tabelas
if tabela_cont:
    display(HTML('<div class="header">📊 MÉTRICAS CONTÍNUAS (bootstrap + Rubin) - total_bill, tip</div>'))
    df_cont = pd.DataFrame(tabela_cont)
    display(df_cont.style.set_table_attributes('class="dataframe-table"'))
if tabela_cat:
    display(HTML('<div class="header">📊 MÉTRICAS ORDINAIS (size, day, time, smoker, sex) + SMAE_ord</div>'))
    df_cat = pd.DataFrame(tabela_cat)
    display(df_cat.style.set_table_attributes('class="dataframe-table"'))
if tabela_extra:
    display(HTML('<div class="header">📊 MÉTRICAS ADICIONAIS (Gower, erro matriz correlação)</div>'))
    df_extra = pd.DataFrame(tabela_extra)
    display(df_extra.style.set_table_attributes('class="dataframe-table"'))

# Gráficos (opcional)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("whitegrid")
    props_percent = [f"{p*100:.0f}%" for p in proporcoes_teste]

    # Extrair médias e intervalos das tabelas
    acc_means = [float(tabela_cat[i]['Acurácia'].split()[0].replace('%',''))/100 for i in range(len(proporcoes_teste))]
    acc_lower = [float(tabela_cat[i]['Acurácia'].split('[')[1].split('%')[0])/100 for i in range(len(proporcoes_teste))]
    acc_upper = [float(tabela_cat[i]['Acurácia'].split('–')[1].split('%')[0])/100 for i in range(len(proporcoes_teste))]
    rmse_means = [float(tabela_cont[i]['RMSE'].split()[0]) for i in range(len(proporcoes_teste))]
    rmse_lower = [float(tabela_cont[i]['RMSE'].split('[')[1].split('–')[0]) for i in range(len(proporcoes_teste))]
    rmse_upper = [float(tabela_cont[i]['RMSE'].split('–')[1].split(']')[0]) for i in range(len(proporcoes_teste))]
    gower_means = [float(tabela_extra[i]['Gower'].split()[0]) for i in range(len(proporcoes_teste))]
    gower_lower = [float(tabela_extra[i]['Gower'].split('[')[1].split('–')[0]) for i in range(len(proporcoes_teste))]
    gower_upper = [float(tabela_extra[i]['Gower'].split('–')[1].split(']')[0]) for i in range(len(proporcoes_teste))]
    smae_cont_means = [float(tabela_cont[i]['SMAE_cont'].split()[0]) for i in range(len(proporcoes_teste))]
    smae_cont_lower = [float(tabela_cont[i]['SMAE_cont'].split('[')[1].split('–')[0]) for i in range(len(proporcoes_teste))]
    smae_cont_upper = [float(tabela_cont[i]['SMAE_cont'].split('–')[1].split(']')[0]) for i in range(len(proporcoes_teste))]
    smae_ord_means = [float(tabela_cat[i]['SMAE_ord'].split()[0]) for i in range(len(proporcoes_teste))]
    smae_ord_lower = [float(tabela_cat[i]['SMAE_ord'].split('[')[1].split('–')[0]) for i in range(len(proporcoes_teste))]
    smae_ord_upper = [float(tabela_cat[i]['SMAE_ord'].split('–')[1].split(']')[0]) for i in range(len(proporcoes_teste))]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    # Acurácia
    axes[0,0].errorbar(props_percent, acc_means, yerr=[np.array(acc_means)-np.array(acc_lower), np.array(acc_upper)-np.array(acc_means)], fmt='o-', capsize=5)
    axes[0,0].set_xlabel('Missing (%)'); axes[0,0].set_ylabel('Acurácia'); axes[0,0].set_title('Acurácia (variáveis ordinais)')
    # RMSE
    axes[0,1].errorbar(props_percent, rmse_means, yerr=[np.array(rmse_means)-np.array(rmse_lower), np.array(rmse_upper)-np.array(rmse_means)], fmt='s-', capsize=5, color='green')
    axes[0,1].set_xlabel('Missing (%)'); axes[0,1].set_ylabel('RMSE'); axes[0,1].set_title('RMSE (variáveis contínuas)')
    # Gower
    axes[1,0].errorbar(props_percent, gower_means, yerr=[np.array(gower_means)-np.array(gower_lower), np.array(gower_upper)-np.array(gower_means)], fmt='^-', capsize=5, color='purple')
    axes[1,0].set_xlabel('Missing (%)'); axes[1,0].set_ylabel('Gower distance'); axes[1,0].set_title('Distância de Gower')
    # SMAE contínuo e ordinal
    axes[1,1].errorbar(props_percent, smae_cont_means, yerr=[np.array(smae_cont_means)-np.array(smae_cont_lower), np.array(smae_cont_upper)-np.array(smae_cont_means)], fmt='d-', capsize=5, label='SMAE contínuo')
    axes[1,1].errorbar(props_percent, smae_ord_means, yerr=[np.array(smae_ord_means)-np.array(smae_ord_lower), np.array(smae_ord_upper)-np.array(smae_ord_means)], fmt='o-', capsize=5, label='SMAE ordinal')
    axes[1,1].set_xlabel('Missing (%)'); axes[1,1].set_ylabel('SMAE'); axes[1,1].set_title('SMAE específico (corrigido)'); axes[1,1].legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    display(HTML(f'<div class="info-box">📈 Gráficos não gerados: {e}</div>'))

# ====================================================================================================
# RESUMO FINAL
# ====================================================================================================
TEMPO_GLOBAL_FIM = time.perf_counter()
TEMPO_GLOBAL_TOTAL = TEMPO_GLOBAL_FIM - TEMPO_GLOBAL_INICIO
display(HTML('<div class="header">✅ RESUMO FINAL - IMPUTAÇÃO COM MICEFOREST (TIPS)</div>'))
display(HTML(f'''
<div class="success-box">
    🔥 EXECUÇÃO CONCLUÍDA!<br>
    ⏱️ Tempo total: {TEMPO_GLOBAL_TOTAL:.2f} segundos ({TEMPO_GLOBAL_TOTAL/60:.2f} min)<br>
    📊 Proporções testadas: {', '.join([f"{p*100:.0f}%" for p in proporcoes_teste])}<br>
    📈 Exemplo (15% miss): Acurácia = {tabela_cat[0]['Acurácia']}, RMSE = {tabela_cont[0]['RMSE']}, SMAE_cont = {tabela_cont[0]['SMAE_cont']}, SMAE_ord = {tabela_cat[0]['SMAE_ord']}
</div>
'''))
print("\n" + "="*80)
print(f"🎉 EXPERIMENTO CONCLUÍDO! Tempo total: {TEMPO_GLOBAL_TOTAL:.2f} segundos ({TEMPO_GLOBAL_TOTAL/60:.2f} minutos)")
print("="*80)